# MOD-06 · NB-04 — Interrupted Time Series & Synthetic Control
### Health Analytics with Python · Module 06: Causal Inference in Health Research
---
**Learning objectives**
- Understand ITS as a quasi-experimental design for policy evaluation
- Fit segmented regression to estimate level and slope changes
- Assess ITS assumptions: no anticipation, no contemporaneous events
- Build a synthetic control from donor units
- Apply structural break detection with the Chow test
- Compare ITS with and without control group (controlled ITS)

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `statsmodels`, `numpy`, `scipy`


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.optimize import minimize
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod06", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)


## 2. ITS Design — Opioid Prescribing Policy

**Scenario:** A state implemented a prescription drug monitoring programme (PDMP) mandatory check policy in month 25. We measure monthly opioid prescription rate per 1000 patients.

**Segmented regression model:**
```
Y_t = beta0 + beta1*t + beta2*D + beta3*(t - T_int)*D + e_t
```
- `beta1` = pre-intervention trend
- `beta2` = immediate level change at intervention
- `beta3` = change in slope (trend change) post-intervention


In [ ]:
# Simulate monthly opioid prescribing data (48 months)
T_TOTAL  = 48
T_INT    = 24   # intervention at month 25 (index 24)
TRUE_LEVEL_CHANGE = -15.0  # immediate drop in prescriptions
TRUE_SLOPE_CHANGE = -0.8   # additional monthly decline post-intervention
PRE_SLOPE = 0.3            # pre-intervention increasing trend

months = np.arange(1, T_TOTAL+1)
D      = (months > T_INT).astype(int)       # post-intervention indicator
t_post = np.maximum(0, months - T_INT)       # time since intervention

# True outcome
np.random.seed(42)
# Add AR(1) autocorrelation
errors = np.zeros(T_TOTAL)
errors[0] = np.random.normal(0, 3)
for i in range(1, T_TOTAL):
    errors[i] = 0.4*errors[i-1] + np.random.normal(0, 2.5)

rx_rate = (80 + PRE_SLOPE*months + TRUE_LEVEL_CHANGE*D
           + TRUE_SLOPE_CHANGE*t_post*D + errors)

df_its = pd.DataFrame({"month":months,"D":D,"t_post":t_post,"rx_rate":rx_rate})
print(f"ITS dataset: {T_TOTAL} months | Intervention at month {T_INT+1}")
print(f"Pre-period mean: {rx_rate[:T_INT].mean():.1f} | Post-period mean: {rx_rate[T_INT:].mean():.1f}")


In [ ]:
# Segmented regression
model_its = smf.ols("rx_rate ~ month + D + t_post", data=df_its).fit()
print(model_its.summary().tables[1])

# Extract key parameters
b0 = model_its.params["Intercept"]
b1 = model_its.params["month"]
b2 = model_its.params["D"]
b3 = model_its.params["t_post"]

print(f"\nPre-intervention slope  : {b1:.3f} rx/1000/month (true: {PRE_SLOPE:.1f})")
print(f"Level change at policy  : {b2:.3f} rx/1000 (true: {TRUE_LEVEL_CHANGE:.1f})")
print(f"Slope change post-policy: {b3:.3f} rx/1000/month (true: {TRUE_SLOPE_CHANGE:.1f})")

# Counterfactual
cf_post = b0 + b1*months
fitted  = model_its.fittedvalues

fig, ax = plt.subplots(figsize=(12,5))
ax.scatter(months, rx_rate, alpha=0.6, s=25, color="gray", label="Observed", zorder=3)
ax.plot(months[:T_INT], fitted[:T_INT], "-", color="#4878CF", lw=2.5, label="Fitted (pre)")
ax.plot(months[T_INT:], fitted[T_INT:], "-", color="#D65F5F", lw=2.5, label="Fitted (post)")
ax.plot(months[T_INT:], cf_post[T_INT:], "--", color="#4878CF", lw=2, alpha=0.7, label="Counterfactual")
ax.fill_between(months[T_INT:], fitted[T_INT:], cf_post[T_INT:],
                alpha=0.15, color="#D65F5F", label=f"Treatment effect")
ax.axvline(T_INT+0.5, color="black", ls="--", lw=1.5, label="PDMP mandatory check")
ax.set_xlabel("Month"); ax.set_ylabel("Opioid Rx rate per 1,000 patients")
ax.set_title("Interrupted Time Series: PDMP Mandatory Check Policy")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.savefig("/tmp/mod06/its_segmented.png",bbox_inches="tight"); plt.show()

# Cumulative effect at end of follow-up
total_effect = b2 + b3*(T_TOTAL - T_INT)
print(f"\nCumulative effect by month {T_TOTAL}: {total_effect:.1f} rx/1000")
print(f"Projected over 12 months post-policy: {b2 + b3*12:.1f} rx/1000")


## 3. Controlled ITS with Comparison Series

In [ ]:
# Controlled ITS: subtract control group trajectory to remove confounders
# Control group: neighbouring state without PDMP policy
np.random.seed(99)
control_errors = np.zeros(T_TOTAL)
control_errors[0] = np.random.normal(0, 3)
for i in range(1, T_TOTAL):
    control_errors[i] = 0.4*control_errors[i-1] + np.random.normal(0, 2.5)

# Control state: similar pre-trend, NO treatment effect
rx_control = 75 + PRE_SLOPE*months + control_errors

df_its["rx_control"] = rx_control
df_its["diff"] = df_its["rx_rate"] - df_its["rx_control"]

# ITS on the difference series
model_diff = smf.ols("diff ~ month + D + t_post", data=df_its).fit()
b2_ctrl = model_diff.params["D"]
b3_ctrl = model_diff.params["t_post"]

print("Controlled ITS (treatment - control):")
print(f"  Level change   : {b2_ctrl:.3f} (true: {TRUE_LEVEL_CHANGE:.1f})")
print(f"  Slope change   : {b3_ctrl:.3f} (true: {TRUE_SLOPE_CHANGE:.1f})")

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(months, rx_rate, "o-", color="#D65F5F", lw=1.5, ms=4, label="Treated state (PDMP)")
axes[0].plot(months, rx_control, "o-", color="#4878CF", lw=1.5, ms=4, label="Control state (no PDMP)")
axes[0].axvline(T_INT+0.5, color="black", ls="--", lw=1.2)
axes[0].set_title("Treated vs Control state"); axes[0].legend(fontsize=8)
axes[0].set_ylabel("Opioid Rx rate"); axes[0].set_xlabel("Month")

axes[1].plot(months, df_its["diff"], "o-", color="gray", lw=1, ms=4, alpha=0.7, label="Difference (T-C)")
axes[1].plot(months, model_diff.fittedvalues, "-", color="#6ACC65", lw=2.5, label="Fitted difference")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].axvline(T_INT+0.5, color="black", ls="--", lw=1.2)
axes[1].set_title("Controlled ITS: Difference series"); axes[1].legend(fontsize=8)
axes[1].set_ylabel("Treated - Control"); axes[1].set_xlabel("Month")
plt.tight_layout(); plt.savefig("/tmp/mod06/controlled_its.png",bbox_inches="tight"); plt.show()


## 4. Synthetic Control Method

In [ ]:
# Synthetic control: find weighted combination of donor states that
# best matches the treated state's pre-intervention outcome trajectory

np.random.seed(42)
N_DONORS  = 8
T_PRE     = T_INT      # pre-intervention period
T_POST    = T_TOTAL - T_INT

# Simulate donor states (each with different base rates and trends)
donor_baselines = np.random.uniform(60, 100, N_DONORS)
donor_trends    = np.random.uniform(0.1, 0.5, N_DONORS)
donor_noise     = np.random.normal(0, 2, (N_DONORS, T_TOTAL))
donors = np.array([donor_baselines[i] + donor_trends[i]*months + donor_noise[i]
                   for i in range(N_DONORS)])  # shape: (N_DONORS, T_TOTAL)

# True weights that would reconstruct treated pre-trend
# (in practice, we solve an optimisation problem)
def synth_loss(weights, donors_pre, treated_pre):
    # weights must sum to 1 and be non-negative
    synth = weights @ donors_pre
    return np.sum((synth - treated_pre)**2)

from scipy.optimize import minimize
treated_pre = rx_rate[:T_PRE]
donors_pre  = donors[:, :T_PRE]

constraints = [{"type":"eq","fun":lambda w: w.sum()-1}]
bounds = [(0,1)]*N_DONORS
w0 = np.ones(N_DONORS)/N_DONORS
result = minimize(synth_loss, w0, args=(donors_pre, treated_pre),
                  method="SLSQP", bounds=bounds, constraints=constraints)
w_opt = result.x
print(f"Optimal weights: {[f'{w:.3f}' for w in w_opt]}")
print(f"Weight sum: {w_opt.sum():.4f}")

# Construct synthetic control
synth_control = w_opt @ donors  # shape: (T_TOTAL,)
# Pre-period fit quality
rmspe_pre = np.sqrt(np.mean((rx_rate[:T_PRE] - synth_control[:T_PRE])**2))
print(f"Pre-period RMSPE: {rmspe_pre:.3f} (lower = better synthetic control fit)")

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(months, rx_rate, "o-", color="#D65F5F", lw=2, ms=5, label="Treated state")
ax.plot(months, synth_control, "s--", color="#4878CF", lw=2, ms=5, label="Synthetic control")
ax.axvline(T_INT+0.5, color="black", ls="--", lw=1.5, label="Policy intervention")
ax.fill_between(months[T_INT:], synth_control[T_INT:], rx_rate[T_INT:],
                alpha=0.2, color="#D65F5F", label="Treatment effect (ATT)")
ax.set_xlabel("Month"); ax.set_ylabel("Opioid Rx rate")
ax.set_title(f"Synthetic Control Method (pre-RMSPE={rmspe_pre:.2f})")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/mod06/synthetic_control.png",bbox_inches="tight"); plt.show()

# ATT estimate post-policy
att_synth = (rx_rate[T_INT:] - synth_control[T_INT:]).mean()
print(f"\nATT estimate (synthetic control): {att_synth:.2f} rx/1000/month")
print(f"True average post-policy effect : {(TRUE_LEVEL_CHANGE + TRUE_SLOPE_CHANGE*np.arange(1,T_POST+1)/2).mean():.2f}")


## 5. Knowledge Check
1. The ITS model finds a slope change of -0.8 but the level change is not significant (p=0.14). How do you interpret the policy's effect?
2. A contemporaneous event (economic recession) began at month 25, the same time as the PDMP policy. How does this threaten ITS validity?
3. The synthetic control has pre-period RMSPE=8.5. Is this adequate? What does a high RMSPE imply?
4. With only 12 pre-intervention time points, what problems arise for the ITS analysis?
5. Implement a permutation test for the synthetic control: swap the treated state with each donor and compute the post/pre RMSPE ratio.


In [ ]:
# Exercise 5 — Synthetic control permutation test (placebo inference)
rmspe_ratios = []
# For each donor, pretend it's the treated state
for d in range(N_DONORS):
    treated_d  = donors[d]  # placebo treated
    donors_d   = np.delete(donors, d, axis=0)  # remaining donors
    donors_pre_d = donors_d[:, :T_PRE]
    treated_pre_d = treated_d[:T_PRE]
    res_d = minimize(synth_loss, np.ones(N_DONORS-1)/(N_DONORS-1),
                     args=(donors_pre_d, treated_pre_d),
                     method="SLSQP", bounds=[(0,1)]*(N_DONORS-1),
                     constraints=[{"type":"eq","fun":lambda w:w.sum()-1}])
    w_d = res_d.x
    synth_d = w_d @ donors_d
    rmspe_pre_d  = np.sqrt(np.mean((treated_d[:T_PRE] - synth_d[:T_PRE])**2))
    rmspe_post_d = np.sqrt(np.mean((treated_d[T_PRE:] - synth_d[T_PRE:])**2))
    ratio_d = rmspe_post_d/rmspe_pre_d if rmspe_pre_d>0 else np.nan
    rmspe_ratios.append(ratio_d)

# True treated state ratio
rmspe_post = np.sqrt(np.mean((rx_rate[T_INT:] - synth_control[T_INT:])**2))
ratio_treated = rmspe_post/rmspe_pre

print("Synthetic Control Permutation Test (post/pre RMSPE ratios):")
print(f"  Treated state ratio : {ratio_treated:.2f}")
print(f"  Donor ratios        : {[f'{r:.2f}' for r in rmspe_ratios]}")
p_val = np.mean([r >= ratio_treated for r in rmspe_ratios if not np.isnan(r)])
print(f"  Permutation p-value : {p_val:.3f}")
print(f"  Inference: {'Significant' if p_val < 0.10 else 'Not significant'} at 10% level")


---
**Next -> NB-05: Instrumental Variables**